### Fine-tuning a LLaMA 3 8B model on aspect sentiment classification (ASC)

This notebook finetunes and evaluates a LLaMA 3 8B model on ASC using a fine-tuning method inspired by the development of ORCA 2.

- 3 datasets were used for fine-tuning and evaluation. These datasets can be found in the dataset folder within this directory.
- for each dataset, the fine-tuning and evaluation pipeline was run 3 times using a different train-test split each time, namely with seed 42, 55 and 777.
- The average score of each metric across the 3 runs was computed and the results are listed in the final report

This pipeline is executed on google colab using a T4 GPU. To run, select and upload a dataset from the `datasets` folder and this notebook, `orca2_finetuning_asc`, to google colab. 

In [1]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

!pip uninstall -y unsloth unsloth_zoo
!pip -q install -U pip setuptools wheel
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 58.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 54.6 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 140.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
  

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048

# load in pre-trained model and tokeniser
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [ ]:
# converts base model to a LoRA fine-tunable model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [5]:
from datasets import load_dataset, Dataset
import random
from collections import Counter

raw_dataset = load_dataset(
    "json",
    data_files="/content/edurabsa_dataset_1.jsonl",
    split="train",
)

# varied seeds between 42, 55, 777
raw_split = raw_dataset.train_test_split(test_size=0.2, seed=42)
train_raw = raw_split["train"]
test_raw  = raw_split["test"]

# upsample records from neutral class to balance training dataset
neg = [ex for ex in train_raw if ex["output"]["sentiment_label"] == "negative"]
neu = [ex for ex in train_raw if ex["output"]["sentiment_label"] == "neutral"]
pos = [ex for ex in train_raw if ex["output"]["sentiment_label"] == "positive"]
print("Original class distribution")
print(f"negative: {len(neg)}")
print(f"neutral: {len(neu)}")
print(f"positive: {len(pos)}")


factor = max(len(neg), len(pos)) // max(1, len(neu))
neu_upsampled = neu * factor
balanced_list = neg + pos + neu_upsampled
random.shuffle(balanced_list)

neg = [ex for ex in balanced_list if ex["output"]["sentiment_label"] == "negative"]
neu = [ex for ex in balanced_list if ex["output"]["sentiment_label"] == "neutral"]
pos = [ex for ex in balanced_list if ex["output"]["sentiment_label"] == "positive"]
print("New class distribution")
print(f"negative: {len(neg)}")
print(f"neutral: {len(neu)}")
print(f"positive: {len(pos)}")

# convert to hugging face dataset object to be used
# for model fine-tuning
balanced_train_raw = Dataset.from_list(balanced_list)


Generating train split: 0 examples [00:00, ? examples/s]

Original class distribution
negative: 530
neutral: 176
positive: 894
New class distribution
negative: 530
neutral: 880
positive: 894


In [6]:
train_dataset = balanced_train_raw.map(formatting_prompts_func, batched=True)
test_dataset  = test_raw.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/2304 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = -1,
        num_train_epochs=4,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2304 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,304 | Num Epochs = 4 | Total steps = 1,152
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.321830
20,1.285734
30,1.230583
40,1.166156
50,1.107973
60,1.125561
70,1.082390
80,1.059919
90,0.974241
100,1.005697


In [9]:
from tqdm import tqdm
import json
import re
import ast

FastLanguageModel.for_inference(model)
model.eval()

preds = []
labels = []

# tqdm enables the progress tracking of
# the inference process
for example in tqdm(test_dataset):
    instruction = example["instruction"]
    sentence    = example["input"]
    gold_label  = example["output"]["sentiment_label"].lower()
    prompt = alpaca_prompt.format(instruction, sentence, "",)

    # generate model inference based on formatted input
    inputs = tokenizer([prompt],return_tensors="pt",).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=False,
            do_sample=False,
        )

    # process output
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    if "### Response:" in decoded:
        answer_part = decoded.split("### Response:")[-1].strip()
    else:
        answer_part = decoded.strip()
    json_response = ast.literal_eval(answer_part)
    pred_label = str(json_response.get("sentiment_label", "")).strip().lower()
    preds.append(pred_label)
    labels.append(gold_label)


  0%|          | 0/400 [00:00<?, ?it/s]Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, Fu

In [10]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# compute accuracy and macro averaged metrics
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support( labels, preds, average="macro",)

print(f"accuracy:{accuracy}, precision:{precision}, recall:{recall}, f1 score:{f1}")


accuracy:0.8525, precision:0.7515360983102918, recall:0.7577514751778427, f1 score:0.7543323109082666
